# NB_lab_test_trends — Silver to Gold

Builds `azuredatabricks_medical.gold.lab_test_trends` — one row per test type per calendar month, with abnormal/critical rates and month-over-month trend.

**Load type:** FULL (CREATE OR REPLACE TABLE … AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE azuredatabricks_medical.gold.lab_test_trends AS
WITH base AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                   AS year,
        MONTH(lr.test_date)                                                  AS month,
        c.month_name,
        c.quarter_name,
        COUNT(*)                                                             AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)           AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)           AS critical_test_count
    FROM azuredatabricks_medical.silver.lab_results lr
    LEFT JOIN azuredatabricks_medical.silver.calendar c
        ON c.date_key = CAST(DATE_FORMAT(DATE_TRUNC('month', lr.test_date), 'yyyyMMdd') AS INT)
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date), c.month_name, c.quarter_name
)
SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2))  AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2))  AS critical_rate_pct,
    CAST(LAG(ROUND(abnormal_test_count / total_tests * 100, 2))
        OVER (PARTITION BY test_name ORDER BY year, month) AS DECIMAL(5,2))  AS prev_month_abnormal_rate_pct,
    CAST(ROUND(abnormal_test_count / total_tests * 100, 2)
        - LAG(ROUND(abnormal_test_count / total_tests * 100, 2))
          OVER (PARTITION BY test_name ORDER BY year, month) AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM base

In [0]:
updated_df = spark.table("azuredatabricks_medical.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))